# Stage 2 - Clustering for diversity

Separate embeddings into clusters -> group 'similar' images together. The query selection chooses the most typical point in each cluster -> multiple points are not taken from the same cluster, as this would be 'redundant' -> eat into the budget unnecessarily.

NB: the latter stages of the TypiClust -> TCP_RP algorithm will use the model + the embeddings it learnt in stage 1. Ensure you've run the first notebook ->  saved the embeddings and model before you run this notebook.

# Step 1: Set up

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors
import numpy as np

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Step 2: Define clustering algorithm

In [ ]:
def kmeans_cluster(embeddings, K):
    """
    embeddings: numpy array of shape (N, 512)
    K: number of clusters = min(|L_prev| + B, max_clusters)
    returns: cluster assignment (labels -> indices) for each of the N points
    """
    # Paper uses KMeans for K<=50, MiniBatchKMeans otherwise (Appendix F.1)
    if K <= 50:
        kmeans = KMeans(n_clusters=K, random_state=42)
    else:
        kmeans = MiniBatchKMeans(n_clusters=K, random_state=42)

    cluster_assignments = kmeans.fit_predict(embeddings)  # (N,) array
    return cluster_assignments

# Stage 3 - Querying for typicality + diversity

# Step 1: define typicality function

> To capture the principle of max density, we define the Typicality of an example by its density in some semantically meaningful feature space. Formally, we measure an example’s Typicality by the inverse of the average Euclidean distance to its K* nearest neighbours, namely:
>
> $$\text{Typicality}(x) = \left(\frac{1}{K}\sum_{x_i \in \text{K-NN}(x)} \|x - x_i\|_2\right)^{-1}$$

*K = 20 was used in the paper, so it will be used in this notebook. However, the paper notes that the value of K is not that important.

In [ ]:
def Typicality(cluster_embeddings, K=20):
    """
    cluster_embeddings: (M, 512) array of embeddings for points in ONE cluster
    K: number of nearest neighbours, paper uses K=20
    returns: (M,) array of typicality scores, one per point
    """

    # Fit KNN on the cluster embeddings
    # K+1 because each point is its own nearest neighbour
    k = min(K + 1, len(cluster_embeddings))
    neighbours = NearestNeighbors(n_neighbors=k).fit(cluster_embeddings)
    distances, _ = neighbours.kneighbors(cluster_embeddings)

    # distances[:, 0] is always 0 (self), so skip it
    avg_distances = distances[:, 1:].mean(axis=1)  # (M,)

    typicality = 1.0 / (avg_distances + 1e-8) # prevent divion by zero error
    return typicality

# Step 2: define query selection algorithm

In [ ]:
def query_selection(embeddings, cluster_assignments, labelled_indices, B):
    """
    embeddings:          (N, 512) numpy array - all 50k embeddings
    cluster_assignments: (N,) numpy array - cluster id for each image
    labelled_indices:    list of ints - indices of images already labelled
    B:                   int - how many new points to query this round (budget)

    returns: list of B indices to query
    """
    # 1. Find uncovered clusters
    covered = set(cluster_assignments[i] for i in labelled_indices)
    K = cluster_assignments.max() + 1  # total number of clusters
    uncovered = [c for c in range(K) if c not in covered]

    # 2. Sort uncovered by cluster size, take B largest (ignore clusters with fewer than 5 points)
    cluster_sizes = {c: np.sum(cluster_assignments == c) for c in uncovered}
    valid_uncovered = [c for c in uncovered if cluster_sizes[c] >= 5]
    largest_B = sorted(valid_uncovered, key=lambda c: -cluster_sizes[c])[:B]

    # 3. For each selected cluster, pick most typical point
    queries = []
    for cluster_id in largest_B:
        cluster_indices = np.where(cluster_assignments == cluster_id)[0] # indices of images in this cluster
        cluster_embs = embeddings[cluster_indices] # embeddings of the images in this cluster

        K_neighbours = min(20, len(cluster_embs))
        typicality_scores = Typicality(cluster_embs, K=K_neighbours) # array of all typicality scores of the embeddings in this cluster
        best_local = np.argmax(typicality_scores) # best typicality scores (indexed locally i.e. indices of typicality scores == cluster_indices == cluster_embs)
        best_global = cluster_indices[best_local] # get global/actual index of image
        queries.append(best_global)

    return queries

In [ ]:
# Define margin and uncertainty query selection algorithms too (for baseline)

def query_margin(linear, labelled_indices, B):
    """
    Selects B points from the unlabelled pool with the lowest softmax margin
    (difference between top-2 class probabilities). Lower margin = more uncertain.
    """
    all_indices = np.arange(len(train_labels))
    unlabelled_mask = np.ones(len(train_labels), dtype=bool)
    unlabelled_mask[labelled_indices] = False
    unlabelled_indices = all_indices[unlabelled_mask]

    X_pool = train_embeddings[unlabelled_indices].to(device)
    with torch.no_grad():
        logits = linear(X_pool)
        probs = torch.softmax(logits, dim=1)  # (M, 10)
        top2 = probs.topk(2, dim=1).values   # (M, 2)
        margins = (top2[:, 0] - top2[:, 1]).cpu().numpy()  # (M,)

    # Select B indices with LOWEST margin (most uncertain)
    lowest_margin_local = np.argsort(margins)[:B]
    return unlabelled_indices[lowest_margin_local].tolist()

def query_uncertainty(linear, labelled_indices, B):
    """
    Selects B points with lowest max softmax probability (least confident).
    """
    all_indices = np.arange(len(train_labels))
    unlabelled_mask = np.ones(len(train_labels), dtype=bool)
    unlabelled_mask[labelled_indices] = False
    unlabelled_indices = all_indices[unlabelled_mask]

    X_pool = train_embeddings[unlabelled_indices].to(device)
    with torch.no_grad():
        logits = linear(X_pool)
        probs = torch.softmax(logits, dim=1)
        max_probs = probs.max(dim=1).values.cpu().numpy()  # (M,)

    # Select B indices with LOWEST max prob (most uncertain)
    lowest_conf_local = np.argsort(max_probs)[:B]
    return unlabelled_indices[lowest_conf_local].tolist()

# TCP_RP Final Implementation

NB: the latter stages of the TypiClust -> TCP_RP algorithm will use the model + the embeddings it learnt in stage 1. Ensure you've run the first notebook ->  saved the embeddings and model before you run this section.

# Step 1: Set up

In [ ]:
## SimCLR model def
# First load the ResNet-18 classifier
resnet18 = torchvision.models.resnet18(weights=None, progress=True)

# Reduce kernel size and stride as CIFAR10 images are very small
resnet18.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)

# Remove max pooling layer for same reason
resnet18.maxpool = nn.Identity()

# Remove final classification layer (as embeddings are in penultimate layer)
resnet18.fc = nn.Identity()

# Define projection head (only used during training)
class ProjectionHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(512, 512)
        self.bn = nn.BatchNorm1d(512)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(512, 128)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.fc2(x)
        return F.normalize(x, dim=1)

# Define SimCLR model (a wrapper of the ResNet-18 and projection head)
class SimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = resnet18
        self.projection = ProjectionHead()

    def forward(self, x1, x2):
        h1 = self.encoder(x1)  # (batch_size, 512)
        h2 = self.encoder(x2)  # (batch_size, 512)

        z1 = self.projection(h1)  # (batch_size, 128)
        z2 = self.projection(h2)  # (batch_size, 128)

        return z1, z2

    # Use after training for embedding extraction
    def get_embedding(self, x):
        return self.encoder(x)

In [ ]:
# Load saved model
checkpoint_path_simclr = '/content/drive/MyDrive/5CCSAMLF_CW2/models/simclr_latest.pth'
checkpoint_simclr = torch.load(checkpoint_path_simclr)

simclr = SimCLR().to(device)
simclr.load_state_dict(checkpoint_simclr['model_state_dict'])

In [ ]:
# Load saved train embeddings + labels
checkpoint_path_train = '/content/drive/MyDrive/5CCSAMLF_CW2/models/train_embeddings.pth'
checkpoint_train = torch.load(checkpoint_path_train)

train_embeddings = checkpoint_train['embeddings']
train_labels = checkpoint_train['labels']

In [ ]:
# Load saved test embeddings + labels
checkpoint_path_test = '/content/drive/MyDrive/5CCSAMLF_CW2/models/test_embeddings.pth'
checkpoint_test = torch.load(checkpoint_path_test)

test_embeddings = checkpoint_test['embeddings']
test_labels = checkpoint_test['labels']

In [ ]:
# Define train/evaluation loop to evaluate TCP_RP AL strategy
def train_and_evaluate(labelled_indices, num_epochs=200):
    """
    Train a linear classifier on the labelled embeddings, evaluate on CIFAR-10 test set.
    Returns (acc, linear) so the model can be reused for query scoring.
    """
    X_train = train_embeddings[labelled_indices].to(device)
    y_train = train_labels[labelled_indices].to(device)
    X_test = test_embeddings.to(device)
    y_test = test_labels.to(device)

    linear = nn.Linear(512, 10).to(device)
    optimiser = torch.optim.SGD(linear.parameters(), lr=2.5,
                                momentum=0.9, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=num_epochs)
    criterion = nn.CrossEntropyLoss()

    linear.train()
    for epoch in range(num_epochs):
        optimiser.zero_grad()
        outputs = linear(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimiser.step()
        scheduler.step()

    linear.eval()
    with torch.no_grad():
        test_outputs = linear(X_test)
        preds = test_outputs.argmax(dim=1)
        acc = (preds == y_test).float().mean().item() * 100
    return acc, linear

# Step 2: Implementation (AL loop)

In [ ]:
'''
For each iteration, new labels are selected to be queried, then a linear classifier is
trained on those labels and evaluated.
'''

N_ROUNDS = 10
N_ITERATIONS = 6
B = 10

accuracies_mat_typiclust = []
accuracies_mat_random = []
accuracies_mat_margin = []
accuracies_mat_uncertainty = []

train_embeddings_np = train_embeddings.numpy()

for round_idx in range(N_ROUNDS):
    print(f"\n--- Round {round_idx+1}/{N_ROUNDS} ---")

    labelled_indices_typi   = []
    labelled_indices_rand   = []
    labelled_indices_margin = []
    labelled_indices_uncert = []

    round_accs_typi   = []
    round_accs_rand   = []
    round_accs_margin = []
    round_accs_uncert = []

    for iteration in range(N_ITERATIONS):
        # ---- TypiClust queries ----
        K = min(len(labelled_indices_typi) + B, 500)
        cluster_assignments = kmeans_cluster(train_embeddings_np, K)
        new_queries_typi = query_selection(train_embeddings_np, cluster_assignments,labelled_indices_typi, B)
        labelled_indices_typi.extend([int(q) for q in new_queries_typi])
        acc_typi, _ = train_and_evaluate(labelled_indices_typi)

        # ---- Random queries ----
        available = list(set(range(len(train_labels))) - set(labelled_indices_rand))
        labelled_indices_rand.extend(
            np.random.choice(available, B, replace=False).tolist()
        )
        acc_rand, _ = train_and_evaluate(labelled_indices_rand)

        # ---- Margin queries ----
        # Iteration 0: no model yet, fall back to cold start
        # NB: set balanced to be True for a class-balnced cold start, and false for a random cold atart
        if len(labelled_indices_margin) == 0:
            labelled_indices_margin = np.random.choice(len(train_labels), B, replace=False).tolist()
        else:
            new_queries_margin = query_margin(linear_margin, labelled_indices_margin, B)
            labelled_indices_margin.extend(new_queries_margin)
        acc_margin, linear_margin = train_and_evaluate(labelled_indices_margin)

        # ---- Uncertainty queries ----
        # Iteration 0: no model yet, fall back to random cold start
        if len(labelled_indices_uncert) == 0:
            labelled_indices_uncert = np.random.choice(len(train_labels), B, replace=False).tolist()
        else:
            new_queries_uncert = query_uncertainty(linear_uncert, labelled_indices_uncert, B)
            labelled_indices_uncert.extend(new_queries_uncert)
        acc_uncert, linear_uncert = train_and_evaluate(labelled_indices_uncert)

        round_accs_typi.append(acc_typi)
        round_accs_rand.append(acc_rand)
        round_accs_margin.append(acc_margin)
        round_accs_uncert.append(acc_uncert)

        print(f"Iteration {iteration+1} | Budget {len(labelled_indices_typi)} | "
              f"TPC_RP: {acc_typi:.1f}% | Rand: {acc_rand:.1f}% | "
              f"Margin: {acc_margin:.1f}% | Uncert: {acc_uncert:.1f}%")

    accuracies_mat_typiclust.append(round_accs_typi)
    accuracies_mat_random.append(round_accs_rand)
    accuracies_mat_margin.append(round_accs_margin)
    accuracies_mat_uncertainty.append(round_accs_uncert)

# Step 3: Results and visualisations

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# Compute mean and std for plot

# Shape: (N_ROUNDS, N_ITERATIONS) -> mean over rounds axis
accuracies_mat_typiclust = np.array(accuracies_mat_typiclust)
accuracies_mat_random = np.array(accuracies_mat_random)
accuracies_mat_margin = np.array(accuracies_mat_margin)
accuracies_mat_uncertainty = np.array(accuracies_mat_uncertainty)

mean_typiclust = accuracies_mat_typiclust.mean(axis=0)  # (6,) - mean per iteration
mean_random = accuracies_mat_random.mean(axis=0)
mean_margin = accuracies_mat_margin.mean(axis=0)
mean_uncertainty = accuracies_mat_uncertainty.mean(axis=0)

std_typiclust = accuracies_mat_typiclust.std(axis=0)   # for error bars in plots
std_random = accuracies_mat_random.std(axis=0)
std_margin = accuracies_mat_margin.std(axis=0)
std_uncertainty = accuracies_mat_uncertainty.std(axis=0)

budgets = [B * (i+1) for i in range(N_ITERATIONS)]  # [10, 20, 30, 40, 50, 60]

In [ ]:
# Form results table

def fmt(means, stds):
    return [f"{m:.3f} ± {s:.3f}" for m, s in zip(means, stds)]

df = pd.DataFrame({
    'Budget': budgets,
    'TPC_RP': fmt(mean_typiclust, std_typiclust),
    'Random': fmt(mean_random, std_random),
    'Margin': fmt(mean_margin, std_margin),
    'Uncertainty': fmt(mean_uncertainty, std_uncertainty),
})

df = df.set_index('Budget')

print(df.to_string())
# print(df.to_latex())

In [ ]:
plt.figure(figsize=(8, 6))

plt.plot(budgets, mean_typiclust, label='TPC_RP')
plt.fill_between(budgets, mean_typiclust - std_typiclust, mean_typiclust + std_typiclust, alpha=0.2)

plt.plot(budgets, mean_random, label='Random')
plt.fill_between(budgets, mean_random - std_random, mean_random + std_random, alpha=0.2)

plt.plot(budgets, mean_margin, label='Margin')
plt.fill_between(budgets, mean_margin - std_margin, mean_margin + std_margin, alpha=0.2)

plt.plot(budgets, mean_uncertainty, label='Uncertainty')
plt.fill_between(budgets, mean_uncertainty - std_uncertainty, mean_uncertainty + std_uncertainty, alpha=0.2)

plt.ylabel('Accuracy (%)')
plt.xlabel('Cumulative Budget')
plt.title('Fully Supervised with Self-Supervised Embedding Framework – CIFAR-10')
plt.grid(True, alpha=0.3)
plt.legend()

plt.savefig('results_with_std.png')
plt.savefig('results_with_std.svg')

plt.show()